# Annotation quality: label agreement, and a re-review of the missed installations

Analysis of the blind re-labelling campaign, drawn by `make_relabel_samples.py` with a fixed seed.

**1. Label agreement on the precision sample.** The metric is Cohen's kappa, **not** a percentage
of agreement, and the distinction matters here. With a base precision around 0.75, an annotator who
answered "true positive" every time would agree with the original labels about 75% of the time
while measuring nothing at all. Kappa corrects for the agreement expected by chance.

Units under audit are oversampled, so that the per-stratum kappa can answer a specific question:
is their over-reported status explained by noisy labels?

**2. Re-review of the missed installations.** This checks the validity of the recall ground truth.
A point re-reviewed as "no installation visible on the imagery" is a false miss, an error in the
ground truth rather than a failure of the detector, and it should be removed.

Note the direction of this test, because it is what makes a clean result meaningful: removing false
misses can only *raise* recall, which *lowers* the correction, which *shrinks* the gaps. The test
therefore runs against the paper's own conclusion. A test that could only help would not be worth
running.

**Expected inputs.** Annotations returned in the label folders, on the same conventions as the
original campaign. The join is spatial, on a centroid within 5 m of the original point, which is
robust to whether the annotation tool preserved the properties; the sample id is used when it
survives the round trip.

**What kappa measures, and what it does not.** A high kappa means the labels are *reproducible*,
not *objectively correct*. A bias shared across both passes, solar thermal read as photovoltaic, a
bright metal roof read as a panel, stays invisible to it. This is the ceiling reachable without
ground truth of a higher order, and it is worth being explicit that it is a ceiling.

In [1]:
# Repository bootstrap: locate the root, then read every data location from paths.py.
import sys
from pathlib import Path


def _repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for parent in [start, *start.parents]:
        if (parent / "paths.py").exists() and (parent / "code").is_dir():
            return parent
    raise FileNotFoundError("run this notebook from inside the repository")


sys.path.insert(0, str(_repo_root()))
import paths


## 0. Configuration, and loading the re-labels

In [2]:
import glob
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

PRECISION_LABELS_DIR = paths.ANNOTATIONS_RAW / 'quality_assessment' / 'precision_labels'
RECALL_LABELS_DIR = paths.ANNOTATIONS_RAW / 'quality_assessment' / 'recall_labels'
KEY_PRECISION = paths.KEY_PRECISION
KEY_RECALL = paths.KEY_RECALL_FN
MATCH_MAX_M = 5.0   # re-label to original point; the squares are 1 m, this is tool margin

POSITIVE_TAGS = {'true', 'normal'}
NEGATIVE_TAGS = {'false'}
LAMBERT93, WGS84 = 'EPSG:2154', 'EPSG:4326'

def load_labels(directory):
    files = sorted(glob.glob(str(directory / '*.geojson')))
    if not files:
        print(f"NO files in {directory}/ -- drop the annotations there and re-run")
        return None
    gdfs = [gpd.read_file(f) for f in files]
    g = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    g = g[g['reviewTag'] != 'unreviewed'].copy()
    # Binary label: 1 confirmed PV, 0 no installation. 'unknown' becomes NaN and is excluded.
    g['relabel'] = np.select([g['reviewTag'].isin(POSITIVE_TAGS), g['reviewTag'].isin(NEGATIVE_TAGS)],
                              [1.0, 0.0], default=np.nan)
    # Geometry to a metric centroid, for the join
    g['geometry'] = g.geometry.to_crs(LAMBERT93) if g.crs else g.geometry
    g = g.set_crs(LAMBERT93, allow_override=True)
    g['geometry'] = g.geometry.centroid
    print(f"{directory}: {len(files)} file(s), {len(g)} annotations "
          f"({g['relabel'].isna().sum()} unknown, excluded)")
    return g

def join_to_key(labels, key_df):
    """Attach each re-label to its original row: by sample id when preserved,
    otherwise by nearest spatial join within MATCH_MAX_M."""
    key = gpd.GeoDataFrame(key_df, geometry=gpd.points_from_xy(key_df['lon'], key_df['lat']),
                            crs=WGS84).to_crs(LAMBERT93)
    if 'sample_id' in labels.columns and labels['sample_id'].notna().all():
        out = key.merge(labels[['sample_id', 'relabel']], on='sample_id', how='inner')
        print(f"joined by sample id: {len(out)} matches")
        return out
    j = gpd.sjoin_nearest(labels[['relabel', 'geometry']], key,
                           how='inner', max_distance=MATCH_MAX_M, distance_col='d')
    j = j[~j.index.duplicated(keep='first')]
    print(f"jointure spatiale (<= {MATCH_MAX_M} m) : {len(j)}/{len(labels)} matches")
    return j

def cohen_kappa(a, b):
    """Cohen's kappa for two aligned binary vectors, NaNs already excluded."""
    a, b = np.asarray(a), np.asarray(b)
    po = (a == b).mean()
    pe = (a.mean() * b.mean()) + ((1 - a.mean()) * (1 - b.mean()))
    return (po - pe) / (1 - pe) if pe < 1 else np.nan


## 1. Agreement on the precision labels

Reading scale (Landis and Koch): above 0.8 is near perfect, 0.6 to 0.8 substantial, 0.4 to 0.6
moderate, below 0.4 a problem. Below roughly 0.6 in a stratum, the labels there are noisy, the
measured precision is not trustworthy, and a full re-labelling of that unit is worth considering.

In [3]:
prec_labels = load_labels(PRECISION_LABELS_DIR)
if prec_labels is not None:
    key_p = pd.read_csv(KEY_PRECISION, dtype={'dpt': str})
    m = join_to_key(prec_labels, key_p)
    m = m.dropna(subset=['relabel'])
    m['relabel'] = m['relabel'].astype(int)

    k_nat = cohen_kappa(m['pred'], m['relabel'])
    agree = (m['pred'] == m['relabel']).mean()
    print(f"NATIONAL : n={len(m)}, accord brut {agree:.1%}, kappa de Cohen = {k_nat:.3f}")

    rows = []
    for stratum, grp in [('uniform', m[m['stratum'] == 'uniform'])] + \
                        [(d, m[(m['stratum'] == 'oversampled') & (m['dpt'] == d)])
                         for d in ['75', '89', '54', '27', '50', '37']]:
        if len(grp) >= 10:
            rows.append({'strate': stratum, 'n': len(grp),
                         'accord': (grp['pred'] == grp['relabel']).mean(),
                         'kappa': cohen_kappa(grp['pred'], grp['relabel']),
                         'precision_orig': grp['pred'].mean(),
                         'precision_relabel': grp['relabel'].mean()})
    kappa_table = pd.DataFrame(rows).round(3)
    display(kappa_table)
    kappa_table.to_csv(paths.KAPPA_RESULTS, index=False)
    print("Ecrit : kappa_precision_results.csv")


/tmp/nbdata/source/annotations/raw/quality_assessment/precision_labels: 1 file(s), 500 annotations (0 unknown, excluded)
joined by sample id: 500 matches
NATIONAL : n=500, accord brut 94.0%, kappa de Cohen = 0.864


,strate,n,accord,kappa,precision_orig,precision_relabel
0,uniform,260,0.958,0.872,0.796,0.785
1,75,40,0.800,0.435,0.175,0.275
2,89,40,0.950,0.896,0.575,0.625
3,54,40,1.000,1.000,0.625,0.625
4,27,40,0.950,0.900,0.500,0.500
5,50,40,0.900,0.763,0.725,0.675
6,37,40,0.925,0.842,0.600,0.625


Ecrit : kappa_precision_results.csv


## 2. Re-review of the missed installations

A re-label of 0, meaning no installation visible on the imagery the model saw, is a **false miss**:
an error in the ground truth, and the point should leave the recall sample.

The adjusted recall is `TP / (TP + FN x (1 - false-miss rate))`. If the national rate exceeds
roughly 5%, the right fix is not this formula but removing the invalidated points and re-running
the chain, since the rate is unlikely to be uniform across units.

In [4]:
fn_labels = load_labels(RECALL_LABELS_DIR)
if fn_labels is not None:
    key_r = pd.read_csv(KEY_RECALL, dtype={'dpt': str})
    mr = join_to_key(fn_labels, key_r)
    mr = mr.dropna(subset=['relabel'])

    faux_fn_nat = 1 - mr['relabel'].mean()   # a re-label of 0 is a false miss
    print(f"NATIONAL: n={len(mr)}, false-miss rate = {faux_fn_nat:.1%}")

    by = mr.assign(faux_fn=1 - mr['relabel']).groupby('stratum')['faux_fn'].agg(['mean', 'count'])
    display(by.round(3))
    by_dpt = mr[mr['stratum'] == 'oversampled'].assign(faux_fn=1 - mr['relabel']) \
        .groupby('dpt')['faux_fn'].agg(['mean', 'count'])
    print("by over-reported unit:")
    display(by_dpt.round(3))

    # Indicative effect on national recall: false misses leave the denominator
    rp = pd.read_csv(paths.RECALL_POINTS, dtype={'dpt': str})
    tp, fn = rp['pred'].sum(), (rp['pred'] == 0).sum()
    r_adj = tp / (tp + fn * (1 - faux_fn_nat))
    print(f"national recall: {tp/(tp+fn):.3f} -> adjusted {r_adj:.3f} "
          f"(assuming a uniform rate; the proper fix is removing the points and re-running)")

    invalid = mr.loc[mr['relabel'] == 0, ['point_id', 'dpt', 'stratum']] if 'point_id' in mr.columns else None
    if invalid is not None and len(invalid):
        invalid.to_csv(paths.FALSE_FN, index=False)
        print(f"wrote false_fn_to_remove.csv ({len(invalid)} points to remove from the recall sample)")

    # False-miss rate by stratum and nationally, consumed by recall.ipynb. The
    # points identified here are removed outright; the rest of the missed
    # population is assumed to carry the same expected rate, so it enters the
    # counts with weight (1 - tau). That is where the fractional recall counts in
    # table.csv come from, and why rounding them to integers would be wrong.
    import json as _json
    tau = {'national': float(faux_fn_nat),
           'by_stratum': {k: float(v) for k, v in
                          mr.assign(f=1 - mr['relabel']).groupby('stratum')['f'].mean().items()},
           'n_reviewed': int(len(mr))}
    _json.dump(tau, open(paths.FN_VALIDITY, 'w'), indent=2)
    print(f"Ecrit : fn_validity.json (tau national = {faux_fn_nat:.3f})")


/tmp/nbdata/source/annotations/raw/quality_assessment/recall_labels: 1 file(s), 250 annotations (0 unknown, excluded)
joined by sample id: 250 matches
NATIONAL: n=250, false-miss rate = 6.8%


,mean,count
stratum,,
oversampled,0.067,120
uniform,0.069,130


by over-reported unit:


,mean,count
dpt,,
27,0.067,15
31,0.067,15
34,0.133,15
37,0.133,15
50,0.067,15
54,0.067,15
81,0.000,15
89,0.000,15


national recall: 0.596 -> adjusted 0.612 (assuming a uniform rate; the proper fix is removing the points and re-running)
wrote false_fn_to_remove.csv (17 points to remove from the recall sample)
Ecrit : fn_validity.json (tau national = 0.068)


## 3. The decision rule, fixed in advance

Stating what each outcome implies before looking at the numbers is what keeps this a test rather
than a description.

**Kappa at least 0.6 everywhere and a false-miss rate below 5%**: labels and ground truth are
reliable, the over-reported statuses must be explained elsewhere, and the paper's figures stand.

**Kappa below 0.6 in a stratum**: re-label that unit and re-run.

**False-miss rate at or above 5%**: remove the invalidated points from the ground truth and re-run
the whole chain. Recall will rise, the correction will shrink and the gaps will narrow. The result
would be a more conservative version of the paper, which is to say a more solid one.